# LeetCode 3369 — Design an Array Statistics Tracker bằng Python

- Link: https://leetcode.com/problems/design-an-array-statistics-tracker/
- Độ khó: **Hard** · **Premium**
- Chủ đề: **Thiết kế cấu trúc dữ liệu**: hàng đợi (FIFO) + tổng cho **trung bình**, **hai heap có xóa lười** cho **trung vị**, **heap (−tần số, giá trị) có kiểm tra lười** cho **mode**

> **Ghi chú Premium:** bài này là Premium nên LeetCode không trả về đề qua API công khai. Đề, ví dụ và tên lớp/phương thức (`StatisticsTracker`, `addNumber`, `removeFirstAddedNumber`, `getMean`, `getMedian`, `getMode`) lấy theo trang doocs (https://leetcode.doocs.org/en/lc/3369/); phần ràng buộc trên trang bị cắt nên notebook giả định cỡ `10^5` lời gọi và giá trị tới `10^9`. Vui lòng **đối chiếu lại** nếu có tài khoản Premium.

Skeleton:

```python
class StatisticsTracker:
    def __init__(self): ...
    def addNumber(self, number: int) -> None: ...
    def removeFirstAddedNumber(self) -> None: ...
    def getMean(self) -> int: ...
    def getMedian(self) -> int: ...
    def getMode(self) -> int: ...
```

## 1. Mục tiêu học

1. Tách bài thiết kế thành ba truy vấn độc lập, mỗi truy vấn một cấu trúc phù hợp.
2. **Trung bình (làm tròn xuống):** giữ tổng và `deque` theo thứ tự thêm (để xóa phần tử thêm sớm nhất).
3. **Trung vị (lấy phần tử giữa lớn hơn khi số phần tử chẵn):** hai heap `lo` (max-heap) và `hi` (min-heap) với **xóa lười** — vì phần tử bị xóa không nằm ở đỉnh heap.
4. **Mode (nhỏ nhất khi hòa):** heap `(−tần số, giá trị)`, chỉ tin đỉnh khi tần số còn khớp — "kiểm tra lười".
5. Kiểm chứng bằng một lớp vét cạn trên hàng nghìn dãy thao tác ngẫu nhiên.

## 2. Đề bài & ví dụ

Thiết kế `StatisticsTracker` lưu một dãy số:

- `addNumber(number)`: thêm số; `removeFirstAddedNumber()`: xóa số được thêm **sớm nhất**;
- `getMean()`: trung bình **làm tròn xuống**; `getMedian()`: phần tử giữa sau khi sắp (nếu có hai, lấy **lớn hơn**); `getMode()`: giá trị xuất hiện nhiều nhất (hòa ⇒ **nhỏ nhất**).

**Ví dụ 1:** thêm `4, 4, 2, 3` ⇒ `getMean = 3`, `getMedian = 4`, `getMode = 4`; xóa phần tử đầu ⇒ `getMode = 2` (dãy `[4, 2, 3]`, hòa ⇒ nhỏ nhất).

**Ví dụ 2:** kết quả `[null, null, null, 7, null, null, null, null, 6, null, 5]`.

In [1]:
import heapq, random, time
from collections import deque, defaultdict, Counter

ex1_ops = ["StatisticsTracker", "addNumber", "addNumber", "addNumber", "addNumber", "getMean", "getMedian", "getMode", "removeFirstAddedNumber", "getMode"]
ex1_args = [[], [4], [4], [2], [3], [], [], [], [], []]
ex1_out = [None, None, None, None, None, 3, 4, 4, None, 2]
ex2_ops = ["StatisticsTracker", "addNumber", "addNumber", "getMean", "removeFirstAddedNumber", "addNumber", "addNumber", "removeFirstAddedNumber", "getMedian", "addNumber", "getMode"]
ex2_args = [[], [9], [5], [], [], [5], [6], [], [], [8], []]
ex2_out = [None, None, None, 7, None, None, None, None, 6, None, 5]

def run(cls, ops, args):
    out, obj = [], None
    for o, a in zip(ops, args):
        if o == 'StatisticsTracker':
            obj = cls(); out.append(None)
        else:
            out.append(getattr(obj, o)(*a))
    return out
print('đã định nghĩa run(cls, ops, args) để chạy chuỗi thao tác kiểu LeetCode')

đã định nghĩa run(cls, ops, args) để chạy chuỗi thao tác kiểu LeetCode


## 3. Kiến thức nền

### 3.1. Trung vị bằng hai heap

Giữ `lo` (nửa nhỏ, max-heap bằng số âm) và `hi` (nửa lớn, min-heap) với `|lo| = ⌊n/2⌋`, `|hi| = n − ⌊n/2⌋`. Khi đó phần tử ở chỉ số `⌊n/2⌋` (0-based) của dãy đã sắp — đúng định nghĩa "lấy giữa lớn hơn" — chính là `min(hi)`.

In [2]:
a = sorted([4, 4, 2, 3])
print(a, '-> chỉ số n//2 =', len(a) // 2, '-> trung vị', a[len(a) // 2])

[2, 3, 4, 4] -> chỉ số n//2 = 2 -> trung vị 4


### 3.2. Xóa lười trong heap

Heap không xóa được phần tử ở giữa trong `O(log n)`. Thay vào đó ghi `delayed[x] += 1`; mỗi khi một phần tử `x` lên **đỉnh** và `delayed[x] > 0` thì bỏ nó đi (giảm `delayed`). Cần đếm **kích thước hợp lệ** riêng (`nlo`, `nhi`) vì độ dài danh sách heap còn chứa phần tử "chết".

### 3.3. Mode bằng heap và kiểm tra lười

Mỗi lần tần số của `x` đổi, đẩy `(−cnt[x], x)` vào heap. Khi truy vấn, đỉnh `(−c, v)` chỉ hợp lệ nếu `cnt[v] == c`; nếu không thì bỏ. Thứ tự `(−tần số, giá trị)` tự động ưu tiên tần số lớn, rồi giá trị nhỏ.

In [3]:
h = []
cnt = Counter()
for x in [4, 4, 2, 3]:
    cnt[x] += 1; heapq.heappush(h, (-cnt[x], x))
print('heap mode:', sorted(h), '-> đỉnh hợp lệ:', h[0])

heap mode: [(-2, 4), (-1, 2), (-1, 3), (-1, 4)] -> đỉnh hợp lệ: (-2, 4)


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1 sau khi xóa số `4` đầu tiên: tần số `{4: 1, 2: 1, 3: 1}`. Heap mode còn các bộ cũ `(−2, 4)` — **không còn đúng** vì `cnt[4] = 1` ⇒ bị bỏ; bộ `(−1, 2)` đúng ⇒ mode `2` (nhỏ nhất trong các giá trị hòa).

Trung vị: `[4, 4, 2, 3]` sắp thành `[2, 3, 4, 4]`, `n//2 = 2` ⇒ `4`.

## 5. Cách 1 — Brute Force: tính lại mỗi truy vấn

In [4]:
class TrackerBrute:
    def __init__(self):
        self.a = deque()
    def addNumber(self, number):
        self.a.append(number)
    def removeFirstAddedNumber(self):
        self.a.popleft()
    def getMean(self):
        return sum(self.a) // len(self.a)
    def getMedian(self):
        return sorted(self.a)[len(self.a) // 2]
    def getMode(self):
        c = Counter(self.a); m = max(c.values())
        return min(v for v in c if c[v] == m)

print(run(TrackerBrute, ex1_ops, ex1_args) == ex1_out, run(TrackerBrute, ex2_ops, ex2_args) == ex2_out)

True True


**Độ phức tạp:** `O(n)` – `O(n log n)` mỗi truy vấn ⇒ `O(q · n log n)` — quá chậm với `10^5` lời gọi.

## 6. Ý tưởng thuật toán / bước đột phá

| Truy vấn | Cấu trúc | Chi phí |
|---|---|---|
| `getMean` | `deque` + tổng | `O(1)` |
| `getMedian` | hai heap + xóa lười | `O(log n)` khấu hao |
| `getMode` | heap `(−cnt, v)` + kiểm tra lười | `O(log n)` khấu hao |

Mỗi phần tử được đẩy/bật khỏi mỗi heap một số hằng lần ⇒ tổng `O(q log q)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Danh sách đã sắp với `bisect` + mode heap lười

`bisect.insort` / `del a[bisect_left(...)]` là `O(n)` (dịch mảng) nhưng chạy bằng C nên nhanh trong thực tế với `n ≤ 10^5`. Trung vị `a[n // 2]` trực tiếp.

In [5]:
import bisect

class TrackerBisect:
    def __init__(self):
        self.q = deque(); self.total = 0; self.sorted = []
        self.cnt = defaultdict(int); self.mh = []
    def addNumber(self, number):
        self.q.append(number); self.total += number
        bisect.insort(self.sorted, number)
        self.cnt[number] += 1; heapq.heappush(self.mh, (-self.cnt[number], number))
    def removeFirstAddedNumber(self):
        x = self.q.popleft(); self.total -= x
        del self.sorted[bisect.bisect_left(self.sorted, x)]
        self.cnt[x] -= 1
        if self.cnt[x]:
            heapq.heappush(self.mh, (-self.cnt[x], x))
    def getMean(self):
        return self.total // len(self.q)
    def getMedian(self):
        return self.sorted[len(self.sorted) // 2]
    def getMode(self):
        while True:
            c, v = self.mh[0]
            if self.cnt[v] == -c:
                return v
            heapq.heappop(self.mh)

print(run(TrackerBisect, ex1_ops, ex1_args) == ex1_out, run(TrackerBisect, ex2_ops, ex2_args) == ex2_out)

True True


### 7.2. Cách 3 — Hai heap xóa lười + mode heap lười (lời giải chính)

In [6]:
class TrackerHeaps:
    def __init__(self):
        self.q = deque(); self.total = 0
        self.lo, self.hi = [], []                 # lo: max-heap (lưu số âm), hi: min-heap
        self.nlo = self.nhi = 0                   # kích thước HỢP LỆ
        self.delayed = defaultdict(int)
        self.cnt = defaultdict(int); self.mh = []

    def _prune(self, h, sign):
        while h and self.delayed[sign * h[0]]:
            self.delayed[sign * h[0]] -= 1
            heapq.heappop(h)

    def _balance(self):
        want = (self.nlo + self.nhi) // 2
        while self.nlo > want:
            v = -heapq.heappop(self.lo); self.nlo -= 1
            heapq.heappush(self.hi, v); self.nhi += 1
            self._prune(self.lo, -1)
        while self.nlo < want:
            v = heapq.heappop(self.hi); self.nhi -= 1
            heapq.heappush(self.lo, -v); self.nlo += 1
            self._prune(self.hi, 1)
        self._prune(self.lo, -1); self._prune(self.hi, 1)

    def addNumber(self, number):
        self.q.append(number); self.total += number
        if self.hi and number >= self.hi[0]:
            heapq.heappush(self.hi, number); self.nhi += 1
        else:
            heapq.heappush(self.lo, -number); self.nlo += 1
        self._balance()
        self.cnt[number] += 1; heapq.heappush(self.mh, (-self.cnt[number], number))

    def removeFirstAddedNumber(self):
        x = self.q.popleft(); self.total -= x
        self.delayed[x] += 1
        if self.lo and x <= -self.lo[0]:
            self.nlo -= 1
        else:
            self.nhi -= 1
        self._prune(self.lo, -1); self._prune(self.hi, 1)
        self._balance()
        self.cnt[x] -= 1
        if self.cnt[x]:
            heapq.heappush(self.mh, (-self.cnt[x], x))

    def getMean(self):
        return self.total // len(self.q)

    def getMedian(self):
        return self.hi[0]

    def getMode(self):
        while True:
            c, v = self.mh[0]
            if self.cnt[v] == -c:
                return v
            heapq.heappop(self.mh)

print(run(TrackerHeaps, ex1_ops, ex1_args) == ex1_out, run(TrackerHeaps, ex2_ops, ex2_args) == ex2_out)

True True


### 7.3. Cách 4 — Danh sách đã sắp chia khối (√n) cho trung vị

Giữ các khối danh sách nhỏ đã sắp (mỗi khối ≤ `B` phần tử). Chèn/xóa tìm khối bằng `bisect` trên giá trị đầu khối, rồi `insort` trong khối `O(B)`; truy vấn phần tử thứ `k` duyệt khối `O(n/B)`. Với `B ≈ 512` đây là ý tưởng của `SortedList` (thư viện `sortedcontainers`) — viết tay để không phụ thuộc thư viện ngoài.

In [7]:
class BlockSorted:
    B = 512
    def __init__(self):
        self.blocks = []; self.n = 0
    def add(self, x):
        if not self.blocks:
            self.blocks.append([x]); self.n = 1; return
        i = bisect.bisect_right([b[0] for b in self.blocks], x) - 1
        i = max(i, 0)
        bisect.insort(self.blocks[i], x); self.n += 1
        if len(self.blocks[i]) > 2 * self.B:
            b = self.blocks[i]; self.blocks[i:i + 1] = [b[:self.B], b[self.B:]]
    def remove(self, x):
        i = bisect.bisect_right([b[0] for b in self.blocks], x) - 1
        while True:                                  # x có thể nằm cuối khối trước khi có giá trị bằng nhau
            b = self.blocks[i]; j = bisect.bisect_left(b, x)
            if j < len(b) and b[j] == x:
                del b[j]; self.n -= 1
                if not b:
                    del self.blocks[i]
                return
            i -= 1
    def kth(self, k):
        for b in self.blocks:
            if k < len(b):
                return b[k]
            k -= len(b)

class TrackerBlocks(TrackerBisect):
    def __init__(self):
        super().__init__(); self.sorted = BlockSorted()
    def addNumber(self, number):
        self.q.append(number); self.total += number; self.sorted.add(number)
        self.cnt[number] += 1; heapq.heappush(self.mh, (-self.cnt[number], number))
    def removeFirstAddedNumber(self):
        x = self.q.popleft(); self.total -= x; self.sorted.remove(x)
        self.cnt[x] -= 1
        if self.cnt[x]:
            heapq.heappush(self.mh, (-self.cnt[x], x))
    def getMedian(self):
        return self.sorted.kth(self.sorted.n // 2)

print(run(TrackerBlocks, ex1_ops, ex1_args) == ex1_out, run(TrackerBlocks, ex2_ops, ex2_args) == ex2_out)

True True


## 8. TRACE từng bước

Trace cách 3 trên ví dụ 2: sau mỗi thao tác in nội dung hợp lệ của hai heap và đỉnh heap mode.

In [8]:
def trace(ops, args):
    obj = None
    for o, a in zip(ops, args):
        if o == 'StatisticsTracker':
            obj = TrackerHeaps(); continue
        r = getattr(obj, o)(*a)
        live = sorted(obj.q)
        k = obj.nlo
        print(f"{o}{tuple(a) if a else '()'} -> {r} | dãy {list(obj.q)}, sắp {live}: lo={live[:k]}, hi={live[k:]}")

trace(ex2_ops, ex2_args)

addNumber(9,) -> None | dãy [9], sắp [9]: lo=[], hi=[9]
addNumber(5,) -> None | dãy [9, 5], sắp [5, 9]: lo=[5], hi=[9]
getMean() -> 7 | dãy [9, 5], sắp [5, 9]: lo=[5], hi=[9]
removeFirstAddedNumber() -> None | dãy [5], sắp [5]: lo=[], hi=[5]
addNumber(5,) -> None | dãy [5, 5], sắp [5, 5]: lo=[5], hi=[5]
addNumber(6,) -> None | dãy [5, 5, 6], sắp [5, 5, 6]: lo=[5], hi=[5, 6]
removeFirstAddedNumber() -> None | dãy [5, 6], sắp [5, 6]: lo=[5], hi=[6]
getMedian() -> 6 | dãy [5, 6], sắp [5, 6]: lo=[5], hi=[6]
addNumber(8,) -> None | dãy [5, 6, 8], sắp [5, 6, 8]: lo=[5], hi=[6, 8]
getMode() -> 5 | dãy [5, 6, 8], sắp [5, 6, 8]: lo=[5], hi=[6, 8]


## 9. Code nộp LeetCode

In [9]:
class StatisticsTracker:
    def __init__(self):
        self.q = deque()
        self.total = 0
        self.lo = []
        self.hi = []
        self.nlo = 0
        self.nhi = 0
        self.delayed = defaultdict(int)
        self.cnt = defaultdict(int)
        self.mh = []

    def _prune(self, h, sign):
        d = self.delayed
        while h and d[sign * h[0]]:
            d[sign * h[0]] -= 1
            heapq.heappop(h)

    def _balance(self):
        want = (self.nlo + self.nhi) // 2
        while self.nlo > want:
            v = -heapq.heappop(self.lo)
            self.nlo -= 1
            heapq.heappush(self.hi, v)
            self.nhi += 1
            self._prune(self.lo, -1)
        while self.nlo < want:
            v = heapq.heappop(self.hi)
            self.nhi -= 1
            heapq.heappush(self.lo, -v)
            self.nlo += 1
            self._prune(self.hi, 1)
        self._prune(self.lo, -1)
        self._prune(self.hi, 1)

    def addNumber(self, number: int) -> None:
        self.q.append(number)
        self.total += number
        if self.hi and number >= self.hi[0]:
            heapq.heappush(self.hi, number)
            self.nhi += 1
        else:
            heapq.heappush(self.lo, -number)
            self.nlo += 1
        self._balance()
        self.cnt[number] += 1
        heapq.heappush(self.mh, (-self.cnt[number], number))

    def removeFirstAddedNumber(self) -> None:
        x = self.q.popleft()
        self.total -= x
        self.delayed[x] += 1
        if self.lo and x <= -self.lo[0]:
            self.nlo -= 1
        else:
            self.nhi -= 1
        self._prune(self.lo, -1)
        self._prune(self.hi, 1)
        self._balance()
        self.cnt[x] -= 1
        if self.cnt[x]:
            heapq.heappush(self.mh, (-self.cnt[x], x))

    def getMean(self) -> int:
        return self.total // len(self.q)

    def getMedian(self) -> int:
        return self.hi[0]

    def getMode(self) -> int:
        mh, cnt = self.mh, self.cnt
        while cnt[mh[0][1]] != -mh[0][0]:
            heapq.heappop(mh)
        return mh[0][1]

print(run(StatisticsTracker, ex1_ops, ex1_args), run(StatisticsTracker, ex2_ops, ex2_args) == ex2_out)

[None, None, None, None, None, 3, 4, 4, None, 2] True


## 10. Vì sao thuật toán đúng

1. **Trung bình:** `deque` giữ đúng thứ tự thêm nên `popleft` xóa đúng số sớm nhất; tổng cập nhật tương ứng; `//` là làm tròn xuống (giá trị dương).
2. **Trung vị:** bất biến — mọi phần tử hợp lệ của `lo` ≤ mọi phần tử hợp lệ của `hi`, `nlo = ⌊n/2⌋`, đỉnh hai heap luôn hợp lệ (đã `_prune`). Khi xóa `x`: nếu `x < max(lo)` thì `x` chắc chắn trong `lo`; nếu `x == max(lo)` ta coi xóa trong `lo` và `_prune(lo)` loại ngay một bản `x` ở đỉnh; ngược lại `x` trong `hi`. ⇒ `nlo`, `nhi` đúng. Khi đó `min(hi)` là phần tử chỉ số `⌊n/2⌋`.
3. **Mode:** heap luôn chứa một bộ đúng `(−cnt[v], v)` cho mọi `v` còn mặt (đẩy mỗi khi đổi); bộ sai lệch bị bỏ khi lên đỉnh. Đỉnh hợp lệ có tần số lớn nhất, hòa thì giá trị nhỏ nhất.

## 11. Phân tích Time / Space Complexity

| Cách | `add`/`remove` | `median` | `mode` | Space |
|---|---|---|---|---|
| 1. Tính lại | `O(1)` | `O(n log n)` | `O(n)` | `O(n)` |
| 2. `bisect` | `O(n)` (C, nhanh) | `O(1)` | `O(log)` khấu hao | `O(n)` |
| 3. Hai heap lười | `O(log n)` khấu hao | `O(1)` | `O(log)` khấu hao | `O(n)` |
| 4. Chia khối | `O(√n)` | `O(√n)` | `O(log)` khấu hao | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề

In [10]:
impls = {'bisect': TrackerBisect, 'heaps': TrackerHeaps, 'blocks': TrackerBlocks, 'submit': StatisticsTracker}
for name, cls in impls.items():
    assert run(cls, ex1_ops, ex1_args) == ex1_out and run(cls, ex2_ops, ex2_args) == ex2_out, name
assert run(TrackerBrute, ex1_ops, ex1_args) == ex1_out
print('Hai ví dụ đề: tất cả đúng.')

Hai ví dụ đề: tất cả đúng.


### 12.2. Dãy thao tác ngẫu nhiên, so với vét cạn

In [11]:
rng = random.Random(3369)
cnt = 0
for _ in range(1500):
    objs = {name: cls() for name, cls in impls.items()}
    ref = TrackerBrute(); size = 0
    for _ in range(50):
        r = rng.random()
        if size == 0 or r < 0.45:
            x = rng.randint(1, rng.choice([3, 6, 10 ** 9]))
            ref.addNumber(x); size += 1
            for o in objs.values(): o.addNumber(x)
        elif r < 0.6:
            ref.removeFirstAddedNumber(); size -= 1
            for o in objs.values(): o.removeFirstAddedNumber()
        else:
            f = rng.choice(['getMean', 'getMedian', 'getMode'])
            exp = getattr(ref, f)()
            for name, o in objs.items():
                assert getattr(o, f)() == exp, (name, f)
            cnt += 1
print('Khớp vét cạn trên', cnt, 'truy vấn ngẫu nhiên.')

Khớp vét cạn trên 29075 truy vấn ngẫu nhiên.


### 12.3. Ca lớn: `10^5` thao tác

In [12]:
def bench(cls, seq):
    o = cls(); t = time.time(); checksum = 0
    for op, x in seq:
        if op == 0: o.addNumber(x)
        elif op == 1: o.removeFirstAddedNumber()
        else: checksum += o.getMean() + o.getMedian() + o.getMode()
    return checksum, time.time() - t

rng = random.Random(1)
seq, size = [], 0
for i in range(10 ** 5):
    r = rng.random()
    if size == 0 or r < 0.5:
        seq.append((0, rng.randint(1, 10 ** 9))); size += 1
    elif r < 0.7 and size > 1:
        seq.append((1, None)); size -= 1
    else:
        seq.append((2, None))
for name in ['heaps', 'bisect', 'blocks']:
    cs, tt = bench(impls[name], seq)
    print(f"{name:7s}: checksum {cs} | {tt:.2f}s")

heaps  : checksum 38098248596303 | 0.11s


bisect : checksum 38098248596303 | 0.20s


blocks : checksum 38098248596303 | 0.22s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Trung vị lấy phần tử giữa **nhỏ hơn** khi số phần tử chẵn

In [13]:
class WrongMedian(TrackerBrute):
    def getMedian(self):
        a = sorted(self.a); return a[(len(a) - 1) // 2]

print('sai :', run(WrongMedian, ex1_ops, ex1_args)[6], '| đúng:', ex1_out[6])

sai : 3 | đúng: 4


**Vì sao sai:** đề yêu cầu lấy giá trị **lớn hơn** trong hai phần tử giữa (`[2, 3, 4, 4]` ⇒ `4`, không phải `3`).

### Lỗi 2 — Mode khi hòa lấy giá trị xuất hiện trước / lớn hơn

In [14]:
class WrongMode(TrackerBrute):
    def getMode(self):
        return Counter(self.a).most_common(1)[0][0]        # hòa: lấy theo thứ tự xuất hiện

print('sai :', run(WrongMode, ex1_ops, ex1_args)[9], '| đúng:', ex1_out[9])

sai : 4 | đúng: 2


**Vì sao sai:** sau khi xóa, dãy `[4, 2, 3]` có ba giá trị hòa; `most_common` trả `4` (xuất hiện trước), đề yêu cầu **nhỏ nhất** (`2`).

### Lỗi 3 — Mode heap không kiểm tra tần số còn đúng

In [15]:
class WrongStaleMode(TrackerHeaps):
    def getMode(self):
        return self.mh[0][1]                                 # SAI: không bỏ bộ đã cũ

print('sai :', run(WrongStaleMode, ex1_ops, ex1_args)[9], '| đúng:', ex1_out[9])

sai : 4 | đúng: 2


**Vì sao sai:** bộ `(−2, 4)` vẫn ở đỉnh dù `4` chỉ còn 1 lần.

### Lỗi 4 — Xóa lười nhưng không giữ kích thước hợp lệ riêng

In [16]:
class WrongSizes(TrackerHeaps):
    def _balance(self):                                     # SAI: cân bằng theo len(heap) (gồm cả phần tử "chết")
        n = len(self.q); want = n // 2
        while len(self.lo) > want:
            heapq.heappush(self.hi, -heapq.heappop(self.lo))
        while len(self.lo) < want and self.hi:
            heapq.heappush(self.lo, -heapq.heappop(self.hi))
        self._prune(self.lo, -1); self._prune(self.hi, 1)

rng = random.Random(4)
found = None
for _ in range(3000):
    ref, w, ops = TrackerBrute(), WrongSizes(), []
    size = 0
    try:
        for _ in range(20):
            if size == 0 or rng.random() < 0.55:
                x = rng.randint(1, 5); ref.addNumber(x); w.addNumber(x); size += 1; ops.append(('add', x))
            else:
                ref.removeFirstAddedNumber(); w.removeFirstAddedNumber(); size -= 1; ops.append(('remove',))
            if size and w.getMedian() != ref.getMedian():
                found = (ops, w.getMedian(), ref.getMedian()); break
    except IndexError:
        found = (ops, 'IndexError', ref.getMedian() if size else None)
    if found:
        break
print('dãy thao tác gây sai:', found[0])
print('sai :', found[1], '| đúng:', found[2])

dãy thao tác gây sai: [('add', 2), ('add', 4), ('add', 1), ('add', 4), ('add', 3), ('remove',), ('remove',), ('add', 5), ('add', 2), ('remove',)]
sai : 3 | đúng: 4


**Vì sao sai:** danh sách heap còn chứa phần tử đã xóa (chưa lên đỉnh), nên `len(heap)` lớn hơn số phần tử hợp lệ; cân bằng theo nó làm lệch ranh giới giữa hai nửa.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Trung vị động có xóa phần tử tùy ý" ⇒ hai heap + **xóa lười** + đếm kích thước hợp lệ (LC 480 Sliding Window Median).
- "Mode động" ⇒ heap `(−tần số, giá trị)` + kiểm tra lười, hoặc bucket theo tần số.
- "Xóa phần tử thêm sớm nhất" ⇒ `deque` lưu thứ tự.
- Bài thiết kế: viết lớp vét cạn và so sánh trên dãy thao tác ngẫu nhiên.

## 15. Biến thể của bài

1. **LC 295 — Find Median from Data Stream:** chỉ thêm, không xóa.
2. **LC 480 — Sliding Window Median:** cửa sổ cố định ⇒ chính là "xóa phần tử sớm nhất".
3. Thêm `getRange()` (max − min) ⇒ hai heap lười riêng cho max và min, hoặc dùng `lo`/`hi`.
4. Mode lớn nhất khi hòa ⇒ đổi khóa heap thành `(−cnt, −v)`.

In [17]:
# Biến thể 2 (LC 480) dùng lại TrackerHeaps; lưu ý LC 480 lấy trung bình hai phần tử giữa khi k chẵn
def sliding_window_median(nums, k):
    t = TrackerHeaps(); out = []
    for i, x in enumerate(nums):
        t.addNumber(x)
        if i >= k:
            t.removeFirstAddedNumber()
        if i >= k - 1:
            if k % 2:
                out.append(float(t.getMedian()))
            else:
                out.append((t.getMedian() + (-t.lo[0])) / 2)
    return out

assert sliding_window_median([1, 3, -1, -3, 5, 3, 6, 7], 3) == [1.0, -1.0, -1.0, 3.0, 5.0, 6.0]
assert sliding_window_median([1, 2, 3, 4, 2, 3, 1, 4, 2], 3) == [2.0, 3.0, 3.0, 3.0, 2.0, 3.0, 2.0]
print('LC 480 ví dụ: đúng.')

LC 480 ví dụ: đúng.


Ghi chú: với `k` chẵn, phần tử giữa nhỏ hơn là `max(lo)` — đã được `_prune` nên đỉnh `lo` luôn hợp lệ.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Ghi chú |
|---|---|---|
| 1 | Tính lại mỗi truy vấn | Oracle |
| 2 | `bisect` danh sách sắp | Ngắn, nhanh trong thực tế |
| 3 | Hai heap lười | **Nộp bài**, `O(log n)` |
| 4 | Danh sách chia khối | Kiểu `SortedList` tự viết |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** thêm `1, 2, 3, 4`; `getMedian()`.

**Bài 2 (tính tay):** thêm `5, 5, 7, 7, 9`; xóa phần tử đầu; `getMode()`, `getMean()`.

**Bài 3 (điền TODO):** thêm phương thức `getRange()` = max − min.

**Bài 4 (tự viết lại):** mode bằng **bucket theo tần số** (`freq → tập giá trị`), lấy giá trị nhỏ nhất của bucket cao nhất.

**Bài 5 (mini challenge):** thêm `getKth(k)` (phần tử nhỏ thứ `k`) dùng cách 4.

In [18]:
def get_range_todo(tracker):
    # TODO
    raise NotImplementedError

class TrackerBucketModeTodo:
    # TODO
    pass

def get_kth_todo(tracker, k):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Sắp `[1,2,3,4]`. (2) `n // 2 = 2`. (3) `3`.

**Bài 2.** (1) Còn `[5, 7, 7, 9]`. (2) Mode `7`. (3) Trung bình `28 // 4 = 7`.

**Bài 3.** (1) Min toàn cục nằm **sâu** trong `lo` (đỉnh `lo` là max của nửa nhỏ), nên không lấy được trực tiếp. (2) Dùng thêm hai heap lười riêng cho min và max. (3) Hoặc danh sách sắp: `a[-1] − a[0]`.

**Bài 4.** (1) `freq[c]` = tập giá trị có tần số `c` (dùng heap/`SortedList` cho mỗi tập). (2) Giữ `maxfreq`. (3) Khi xóa, nếu bucket `maxfreq` rỗng thì `maxfreq −= 1`.

**Bài 5.** (1) `BlockSorted.kth`. (2) Chỉ số 0-based. (3) So với `sorted`.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2
t = StatisticsTracker()
for x in [1, 2, 3, 4]:
    t.addNumber(x)
assert t.getMedian() == 3
t = StatisticsTracker()
for x in [5, 5, 7, 7, 9]:
    t.addNumber(x)
t.removeFirstAddedNumber()
assert t.getMode() == 7 and t.getMean() == 7
print('Bài 1: 3 | Bài 2: mode 7, mean 7')

# Bài 3 — thêm getRange bằng hai heap lười
class TrackerWithRange(TrackerHeaps):
    def __init__(self):
        super().__init__(); self.mn, self.mx = [], []; self.dmn, self.dmx = defaultdict(int), defaultdict(int)
    def addNumber(self, number):
        super().addNumber(number); heapq.heappush(self.mn, number); heapq.heappush(self.mx, -number)
    def removeFirstAddedNumber(self):
        x = self.q[0]; super().removeFirstAddedNumber(); self.dmn[x] += 1; self.dmx[x] += 1
    def getRange(self):
        while self.dmn[self.mn[0]]:
            self.dmn[self.mn[0]] -= 1; heapq.heappop(self.mn)
        while self.dmx[-self.mx[0]]:
            self.dmx[-self.mx[0]] -= 1; heapq.heappop(self.mx)
        return -self.mx[0] - self.mn[0]

rng = random.Random(3)
for _ in range(300):
    ref, t = TrackerBrute(), TrackerWithRange(); size = 0
    for _ in range(30):
        if size == 0 or rng.random() < 0.6:
            x = rng.randint(1, 9); ref.addNumber(x); t.addNumber(x); size += 1
        else:
            ref.removeFirstAddedNumber(); t.removeFirstAddedNumber(); size -= 1
        if size:
            assert t.getRange() == max(ref.a) - min(ref.a)
print('Bài 3 khớp vét cạn.')

Bài 1: 3 | Bài 2: mode 7, mean 7
Bài 3 khớp vét cạn.


In [20]:
# Bài 4, 5
class TrackerBucketMode(TrackerBisect):
    def __init__(self):
        super().__init__(); self.bucket = defaultdict(list); self.maxf = 0
    def addNumber(self, number):
        self.q.append(number); self.total += number; bisect.insort(self.sorted, number)
        c = self.cnt[number]
        if c: self.bucket[c].remove(number)
        self.cnt[number] = c + 1; bisect.insort(self.bucket[c + 1], number)
        self.maxf = max(self.maxf, c + 1)
    def removeFirstAddedNumber(self):
        x = self.q.popleft(); self.total -= x; del self.sorted[bisect.bisect_left(self.sorted, x)]
        c = self.cnt[x]; self.bucket[c].remove(x); self.cnt[x] = c - 1
        if c - 1: bisect.insort(self.bucket[c - 1], x)
        while self.maxf and not self.bucket[self.maxf]:
            self.maxf -= 1
    def getMode(self):
        return self.bucket[self.maxf][0]

def get_kth(tracker, k):
    return tracker.sorted.kth(k)

rng = random.Random(5)
for _ in range(400):
    ref, a, b = TrackerBrute(), TrackerBucketMode(), TrackerBlocks(); size = 0
    for _ in range(40):
        if size == 0 or rng.random() < 0.55:
            x = rng.randint(1, 7); ref.addNumber(x); a.addNumber(x); b.addNumber(x); size += 1
        else:
            ref.removeFirstAddedNumber(); a.removeFirstAddedNumber(); b.removeFirstAddedNumber(); size -= 1
        if size:
            assert a.getMode() == ref.getMode()
            k = rng.randrange(size)
            assert get_kth(b, k) == sorted(ref.a)[k]
print('Bài 4, 5 khớp vét cạn.')

Bài 4, 5 khớp vét cạn.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
q = deque(); total = 0                      # mean = total // len(q)
lo (max-heap), hi (min-heap), nlo = n//2   # median = hi[0]
delayed[x] += 1 khi xóa; prune đỉnh; cân bằng theo nlo/nhi (không theo len)
mh: push (-cnt[x], x) mỗi khi cnt đổi; getMode bỏ đỉnh có cnt sai
```

**Checklist:**

- [ ] Trung vị = phần tử chỉ số `n // 2` (lấy giữa **lớn hơn**)?
- [ ] Mode hòa ⇒ giá trị **nhỏ nhất**?
- [ ] Xóa lười có đếm kích thước hợp lệ riêng?
- [ ] Mode heap kiểm tra tần số còn khớp?
- [ ] Đối chiếu chữ ký nếu có Premium?